# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:
# Drop rows with any missing values
df_clean = spaceship.dropna()

print("Original shape:", spaceship.shape)
print("Clean shape:", df_clean.shape)
print("Rows removed:", spaceship.shape[0] - df_clean.shape[0])

Original shape: (8693, 14)
Clean shape: (6606, 14)
Rows removed: 2087


In [4]:
df_clean["Deck"] = df_clean["Cabin"].str.split("/").str[0]

print(df_clean["Deck"].unique())

['B' 'F' 'A' 'G' 'E' 'C' 'D' 'T']


C:\Users\Utilizador\AppData\Local\Temp\ipykernel_35804\3219191353.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean["Deck"] = df_clean["Cabin"].str.split("/").str[0]


In [5]:
df_clean.drop(columns=["PassengerId", "Name"], inplace=True)

C:\Users\Utilizador\AppData\Local\Temp\ipykernel_35804\2685270498.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean.drop(columns=["PassengerId", "Name"], inplace=True)


In [6]:
df_encoded = pd.get_dummies(df_clean, drop_first=True)

print(df_encoded.head())

    Age  RoomService  FoodCourt  ShoppingMall     Spa  VRDeck  Transported  \
0  39.0          0.0        0.0           0.0     0.0     0.0        False   
1  24.0        109.0        9.0          25.0   549.0    44.0         True   
2  58.0         43.0     3576.0           0.0  6715.0    49.0        False   
3  33.0          0.0     1283.0         371.0  3329.0   193.0        False   
4  16.0        303.0       70.0         151.0   565.0     2.0         True   

   HomePlanet_Europa  HomePlanet_Mars  CryoSleep_True  ...  \
0               True            False           False  ...   
1              False            False           False  ...   
2               True            False           False  ...   
3               True            False           False  ...   
4              False            False           False  ...   

   Destination_PSO J318.5-22  Destination_TRAPPIST-1e  VIP_True  Deck_B  \
0                      False                     True     False    True   
1       

In [7]:
from sklearn.preprocessing import StandardScaler


X = df_encoded.drop("Transported", axis=1)
y = df_encoded["Transported"]


from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [8]:
from sklearn.model_selection import train_test_split


X = df_encoded.drop("Transported", axis=1)
y = df_encoded["Transported"]


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (5284, 5323)
X_test: (1322, 5323)
y_train: (5284,)
y_test: (1322,)


- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)


rf_model.fit(X_train, y_train)


y_pred_rf = rf_model.predict(X_test)


print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_rf))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_rf))

Random Forest Accuracy: 0.7859304084720121

Classification Report:

              precision    recall  f1-score   support

       False       0.78      0.79      0.78       656
        True       0.79      0.79      0.79       666

    accuracy                           0.79      1322
   macro avg       0.79      0.79      0.79      1322
weighted avg       0.79      0.79      0.79      1322


Confusion Matrix:

[[516 140]
 [143 523]]


- Evaluate your model

In [14]:
# 78.5% accuracy so far

In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


rf = RandomForestClassifier(random_state=42)


param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}


grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)


grid_search.fit(X_train, y_train)


best_rf = grid_search.best_estimator_


y_pred_best_rf = best_rf.predict(X_test)


print("Best Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)
print("\nTest Accuracy:", accuracy_score(y_test, y_pred_best_rf))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_best_rf))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_best_rf))

Fitting 5 folds for each of 216 candidates, totalling 1080 fits
Best Parameters: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
Best CV Score: 0.8009111335684184

Test Accuracy: 0.7874432677760969

Classification Report:

              precision    recall  f1-score   support

       False       0.78      0.79      0.79       656
        True       0.79      0.79      0.79       666

    accuracy                           0.79      1322
   macro avg       0.79      0.79      0.79      1322
weighted avg       0.79      0.79      0.79      1322


Confusion Matrix:

[[518 138]
 [143 523]]


**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

- Run Grid Search

- Evaluate your model

In [13]:
# We improved but with no significance value 